In [1]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/dpo/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())

state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)

/root/micromamba/envs/fla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/root/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


<All keys matched successfully>

In [2]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(dtype=dtype, device=device)

In [3]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=512,
    do_sample=True,
    top_k=20,
    top_p=0.7,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.1,
    use_cache=True
)

prompt = [
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot>"
]
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

In [4]:
with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=False,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>**Respondue:**

**Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **Issue:** **

In [5]:
prompt = [
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot>"
]
tokenizer.pad_token_ids = tokenizer.eos_token_ids
model.pad_token_ids = tokenizer.eos_token_ids
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=False,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: </s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>提创意的「�家。年迈时，</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

In [6]:
prompt = [
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot>"
]
tokenizer.pad_token_ids = 3
model.pad_token_ids = 3
inputs = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left")

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(
        input_ids=inputs.input_ids.to(device),
        attention_mask=inputs.attention_mask.to(device),
        use_cache=True,
        use_varlen_inference=True,
        generation_config=generate_config
    )

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>Transformer架构（Generative Adversarial Networks，GAN）是一种基于Transformer架构的神经网络架构，它通过一个卷积神经网络（GAN）来生成输出、预测和生成输出。GAN架构通过一个卷积层（Convolution）层，将输入的输入转换为输入输出，再通过一个卷积层来生成输出。GAN架构的核心是损失函数，它将输入的输出映射到输入层，然后通过损失函数将输出映射到输出层，最终输出输出。
GAN架构的基本思想是通过一个卷积层来生成输出，并将输出映射到输出层。GAN架构的核心是损失函数，它将输出映射到输出层，然后通过损失函数将输出映射到输出层，最终输出输出。
Transformer架构在图像识别、语音识别、自然语言处理、推荐系统等多个领域中发挥着重要作用。</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>